# Day 5 — Structured Outputs & Tool Calling

---

Today: make AI output *machine-readable* and *action-taking*.

1. **Structured outputs** — get JSON your Python code can safely use.
2. **Tool calling** — let the AI *ask your code* to run a function (like `get_weather(city='Paris')`) and use the result.

Tool calling is the single most important skill from this section. It's how ChatGPT calls a calculator, how Copilot edits files, how every AI agent works.

In [ ]:
!pip install openai together python-dotenv --quiet

In [1]:
import os, json
from dotenv import load_dotenv
from openai import OpenAI
from together import Together

load_dotenv()

tg = Together()
oa = OpenAI() if os.getenv("OPENAI_API_KEY") else None

TG_MODEL = "openai/gpt-oss-20b"
OA_MODEL = "gpt-4o-mini"

## 1. Why structured outputs matter

Real apps need *data*, not paragraphs. If you're building an invoice bot you want:

```json
{"amount": 4200, "currency": "USD", "due_date": "2026-01-15"}
```

not:

> *"The invoice is for four thousand two hundred US dollars, due in mid-January of next year."*

Asking the AI to reply in JSON gives you machine-readable data. Two levels of guarantee:

1. **Prompt-only** — "Reply in JSON". Works ~90% of the time.
2. **JSON mode** — the API guarantees the reply parses as JSON. `response_format={"type":"json_object"}`.

In [2]:
text = "Rohan Mehta is a 34-year-old software engineer. Reach him at rohan@example.com."

prompt = f'''Extract the following as JSON with keys: name, age, email.
Reply with ONLY the JSON object, nothing else.

Text: {text}'''

resp = tg.chat.completions.create(
    model=TG_MODEL,
    messages=[{"role": "user", "content": prompt}],
    response_format={"type": "json_object"},
    max_tokens=200,
)

data = json.loads(resp.choices[0].message.content)
print(data)
print("name:", data["name"])
print("age :", data["age"])

{'name': 'Rohan Mehta', 'age': 34, 'email': 'rohan@example.com'}
name: Rohan Mehta
age : 34


Because we set `response_format="json_object"`, we can safely call `json.loads` — no fragile regex, no crashes.

## 2. Tool calling — the big idea

Regular chat: user → AI → text answer.

**Tool calling** adds a step:

1. You tell the AI *"here are some functions you can call"* — as a schema.
2. The AI decides on its own to call one, and sends you the arguments as JSON.
3. YOUR code runs the function.
4. You send the result back to the AI.
5. The AI writes the final answer using the result.

**Key insight:** the AI never actually runs code. It writes a *request* to run code. You choose whether to honor it. That's the entire safety and control model.

## 3. Tool calling — end-to-end example

We'll pretend to have a weather service. Watch the round trip.

(Tool calling works best on OpenAI's models — Together AI's LLaMA supports it too but is less reliable. We'll use OpenAI here.)

In [3]:
# The 'real' function (fake for this demo)
def get_weather(city: str, unit: str = "celsius") -> dict:
    fake = {
        "Paris":     {"temp": 21, "condition": "sunny"},
        "London":    {"temp": 15, "condition": "cloudy"},
        "Bangalore": {"temp": 27, "condition": "humid"},
    }
    data = fake.get(city, {"temp": 20, "condition": "unknown"})
    return {"city": city, "unit": unit, **data}

# The schema we tell the AI about
TOOLS = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["city"],
        },
    },
}]

print("schema ready — the AI now knows the get_weather tool exists.")

schema ready — the AI now knows the get_weather tool exists.


In [ ]:
if tg is None:
    print("[skipping — set OPENAI_API_KEY to run this cell]")
else:
    messages = [
        {"role": "system", "content": "You are a helpful weather assistant."},
        {"role": "user",   "content": "What's the weather like in Paris right now?"},
    ]

    # Round 1 — model decides to call the tool
    r1 = tg.chat.completions.create(model=TG_MODEL, messages=messages, tools=TOOLS)
    msg = r1.choices[0].message
    call = msg.tool_calls[0]

    args = json.loads(call.function.arguments)
    print("Model asked to call:", call.function.name, args)

    # WE run the real function
    result = get_weather(**args)
    print("Function returned  :", result)

    # Round 2 — feed the result back, model writes the final answer
    messages.append(msg)
    messages.append({
        "role": "tool",
        "tool_call_id": call.id,
        "content": json.dumps(result),
    })
    r2 = tg.chat.completions.create(model=TG_MODEL, messages=messages, tools=TOOLS)
    print("\nFinal answer:", r2.choices[0].message.content)

Model asked to call: get_weather {'city': 'Paris', 'unit': 'celsius'}
Function returned  : {'city': 'Paris', 'unit': 'celsius', 'temp': 21, 'condition': 'sunny'}

Final answer: <|channel|>analysis<|message|>We have to answer user: "What's the weather like in Paris right now?" We have the result from function call: temp 21 Celsius, condition sunny. Likely also add humidity, wind? But given data we only have temp and condition. Provide friendly answer.<|end|><|start|>assistant<|channel|>final<|message|>Sure! 📅 **Paris – Current Weather**

- **Temperature:** 21 °C  
- **Condition:** Sunny & clear skies  

It’s a pleasant day for a stroll through the city—just a light jacket if you’re heading out early or if you’re planning a late‑afternoon walk. Enjoy the sunshine! 🌞


**What just happened** (numbered):

1. You asked the AI: *"what's the weather in Paris?"*
2. The AI replied with a JSON request: `get_weather(city='Paris')`.
3. YOUR Python code ran the actual function.
4. You sent the result back.
5. The AI wrote *"It's 21°C and sunny in Paris"*.

Every real AI app that touches the outside world (Zapier, Notion AI, Copilot's file edits) uses this exact pattern.

## 4. Multiple tools — the AI picks the right one

Give it 2+ tools and it will choose based on the question.

In [5]:
TOOLS2 = [
    {"type":"function", "function":{
        "name": "get_weather",
        "description": "Current weather for a city.",
        "parameters": {"type":"object", "properties":{"city":{"type":"string"}}, "required":["city"]},
    }},
    {"type":"function", "function":{
        "name": "get_time",
        "description": "Current local time in a city.",
        "parameters": {"type":"object", "properties":{"city":{"type":"string"}}, "required":["city"]},
    }},
]

QUESTIONS = [
    "What's the weather in Tokyo?",
    "What time is it in Sydney right now?",
]

if tg:
    for q in QUESTIONS:
        r = tg.chat.completions.create(
            model=TG_MODEL,
            messages=[{"role":"user","content":q}],
            tools=TOOLS2,
        )
        msg = r.choices[0].message
        call = msg.tool_calls[0]
        print(f"Q: {q}")
        print(f"  → tool  : {call.function.name}")
        print(f"  → args  : {call.function.arguments}")
        print(f"  → model response: {msg.content!r}")
else:
    print("[skipping — set OPENAI_API_KEY]")

Q: What's the weather in Tokyo?
  → tool  : get_weather
  → args  : {"city": "Tokyo"}
  → model response: 'I’m sorry, but I can’t retrieve that information right now.'
Q: What time is it in Sydney right now?
  → tool  : get_time
  → args  : {"city": "Sydney"}
  → model response: 'It’s currently 02:17\u202fa.m. in Sydney (Australia/Sydney time zone).'


## Recap

- **Structured outputs**: ask for JSON with `response_format={"type":"json_object"}`, then `json.loads`.
- **Tool calling** = the AI writes a *request* to run one of your functions, you run it, send the result back.
- The AI **never executes code** — it just picks tools + arguments. You stay in control.
- Multi-tool: the AI picks the right one based on the question.
- This is how every real-world AI agent (Zapier, Copilot, Cursor) actually gets things done.

Tomorrow: cost control, streaming, and async — the last piece before the capstone.